In [1]:
import bvhio
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import kinematics as kn
import model as mod

from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R
from kinematics import Joint

matplotlib.use('QtAgg')

In [2]:
# Normalization

def facing_correction(root: Joint) -> np.ndarray:
    y_angle = R.from_quat(root.quat).as_euler('yxz')[0]
    return R.from_euler('y', -y_angle).as_quat()

def get_spine_length(root: Joint, spine_chain: list[str]) -> float:
    """Calculates the total length of the spine by summing bone lengths."""
    total_length = 0.0
    
    for i in range(1, len(spine_chain)):
        parent = root[spine_chain[i-1]]
        child  = root[spine_chain[i]]
        
        if parent is None or child is None:
            print(f"Warning: Joint missing in spine chain ({spine_chain[i-1]} or {spine_chain[i]})")
            continue
            
        # The bone length is the physical distance between the parent and child
        # Since child.offset is the local translation from the parent, 
        # its magnitude is exactly the bone length!
        bone_length = np.linalg.norm(child.offset)
        total_length += bone_length
        
    return total_length

def normalize_skeleton_scale(root: Joint, target_spine_length: float = 1.0) -> Joint:
    """
    Scales the entire skeleton proportionally so its spine matches the target length.
    """
    # 1. Define the spine hierarchy exactly as it appears in the BVH file
    SPINE_CHAIN = [
        "root", 
        "torso_1", 
        "torso_2", 
        "torso_3", 
        "torso_4", 
        "torso_5", 
        "torso_6", 
        "torso_7", 
        "neck_1"
    ]
    
    # 2. Get current spine length
    current_length = get_spine_length(root, SPINE_CHAIN)
    if current_length < 1e-6:
        print("Error: Spine length is near zero. Check your joint IDs.")
        return root
        
    # 3. Calculate the universal scalar
    scale_factor = target_spine_length / current_length

    # 4. Apply the scalar to EVERY joint's local offset
    def _scale_walk(joint: Joint):
        joint.offset = joint.offset * scale_factor
        for child in joint.children:
            _scale_walk(child)
            
    _scale_walk(root)

    return root

def normalize(root: Joint) -> Joint:
    root.offset = np.array([0, 0, 0])
    root.quat   = root.transform.rotate(facing_correction(root))
    return normalize_skeleton_scale(root)

In [3]:
# Graph plotting helpers

def extract_joints_and_bones(root: Joint) -> tuple[list[Joint], list[tuple[Joint, Joint]]]:
    """Flatten the Joint tree into a list of joints and parent-child bone pairs."""
    joints = []
    bones  = []

    def _walk(joint: Joint):
        joints.append(joint)
        for child in joint.children:
            bones.append((joint, child))
            _walk(child)

    _walk(root)
    return joints, bones


def extract_positions_optimized(frames: list[Joint], joints: list[Joint]) -> np.ndarray:
    """
    Computes world positions efficiently using a single top-down tree walk per frame.
    Bypasses the expensive recursive HTM .root property.
    """
    joint_count = len(joints)
    frame_count = len(frames)
    positions   = np.zeros((frame_count, joint_count, 3))
    
    # Map joint IDs to flat array indices for O(1) filling
    idx_of = {j.id: i for i, j in enumerate(joints)}

    for f, frame_root in enumerate(frames):
        def _walk_and_compute(joint: Joint, parent_world: np.ndarray):
            # Compute world transform downstream linearly
            local_T = joint.transform.local
            world_T = local_T if parent_world is None else parent_world @ local_T
            
            # Save world position directly
            if joint.id in idx_of:
                positions[f, idx_of[joint.id]] = world_T[:3, 3]
            
            for child in joint.children:
                _walk_and_compute(child, world_T)

        _walk_and_compute(frame_root, None)

    return positions


def plot_skeleton(frames: list[Joint]):
    """Animate a list of per-frame Joint trees with optimized lookups."""
    if not frames:
        return

    # ── 1. Precalculate Topology & Positions Once ─────────────────────
    joints, bones = extract_joints_and_bones(frames[0])
    positions     = extract_positions_optimized(frames, joints)
    frame_count   = len(frames)

    # Precalculate flat array indices for bones to keep the animation loop blazing fast
    idx_of       = {j.id: i for i, j in enumerate(joints)}
    bone_indices = [(idx_of[p.id], idx_of[c.id]) for p, c in bones]

    # ── 2. Figure Setup ───────────────────────────────────────────────
    fig = plt.figure(figsize=(8, 8))
    ax  = fig.add_subplot(111, projection='3d')

    mn  = positions.reshape(-1, 3).min(axis=0)
    mx  = positions.reshape(-1, 3).max(axis=0)
    pad = (mx - mn).max() * 0.1
    
    mid_x = (mx[0] + mn[0]) / 2
    mid_y = (mx[1] + mn[1]) / 2
    mid_z = (mx[2] + mn[2]) / 2
    
    # Find the maximum span across any axis and add padding
    max_range = (mx - mn).max()
    half_extent = (max_range / 2) + pad

    ax.set_xlim(mid_x - half_extent, mid_x + half_extent)
    ax.set_ylim(mid_z - half_extent, mid_z + half_extent)  # Z → matplotlib Y
    ax.set_zlim(mid_y - half_extent, mid_y + half_extent)  # Y → matplotlib Z

    ax.set_box_aspect((1, 1, 1))
    ax.invert_yaxis()
    ax.set_xlabel('X')
    ax.set_ylabel('Z')
    ax.set_zlabel('Y')

    scatter    = ax.scatter([], [], [], c='red', s=20)
    bone_lines = [ax.plot([], [], [], c='black', lw=1)[0] for _ in bones]
    title      = ax.set_title("")

    # ── 3. Optimized Animation Callback ───────────────────────────────
    def update(frame):
        pts = positions[frame]
        
        # Update joint dots (Swapping Y and Z for Matplotlib's coordinate system)
        scatter._offsets3d = (pts[:, 0], pts[:, 2], pts[:, 1])

        # Update bone lines using precalculated integer indices
        for line, (pi, ci) in zip(bone_lines, bone_indices):
            line.set_data([pts[pi, 0], pts[ci, 0]], [pts[pi, 2], pts[ci, 2]])
            line.set_3d_properties([pts[pi, 1], pts[ci, 1]])

        title.set_text(f"Frame {frame}/{frame_count - 1}")
        return [scatter, *bone_lines, title]

    # ani = FuncAnimation(fig, update, frames=frame_count, interval=33, blit=False)
    ani = FuncAnimation(fig, update, frames=frame_count, interval=16, blit=False)
    plt.show()
    return ani

In [4]:
# Graph plotter

print("Loading model and normalizing...")
frames = mod.load_model("../assets/MCPM_20260423_140006.bvh", frameMap=normalize)

print("Plotting...")
ani = plot_skeleton(frames)

Loading model and normalizing...
Plotting...


In [5]:
import cProfile
import pstats

with cProfile.Profile() as pr:
    frames = mod.load_model("../assets/MCPM_20260423_140006.bvh", frameMap=normalize)

stats = pstats.Stats(pr)
stats.sort_stats('cumulative')
stats.print_stats(15)  # top 15 slowest calls

         3218034 function calls (2713984 primitive calls) in 13.407 seconds

   Ordered by: cumulative time
   List reduced from 270 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      2/1    0.012    0.006   13.354   13.354 /home/airgeddon1337/Project/SENDAI/dance-notation-etl/src/model.py:35(load_model)
     1813    0.036    0.000    7.707    0.004 /tmp/nix-shell.hUlcMi/ipykernel_193208/239439639.py:63(normalize)
     1813    0.056    0.000    3.636    0.002 /home/airgeddon1337/Project/SENDAI/dance-notation-etl/src/model.py:24(convertBvhToHierarchy)
     1813    0.014    0.000    3.621    0.002 /tmp/nix-shell.hUlcMi/ipykernel_193208/239439639.py:27(normalize_skeleton_scale)
47138/5439    0.696    0.000    3.477    0.001 /home/airgeddon1337/Project/SENDAI/dance-notation-etl/src/model.py:16(_build_tree)
520331/152292    2.265    0.000    2.882    0.000 /home/airgeddon1337/Project/SENDAI/dance-notation-etl/src/kinematics.py:14(_in